# 8.9 切回一套修正，需要恢復哪些東西？


同一個基礎模型配兩套LoRA修正，一套教短答、一套教例子。切回短答時，需要恢復整套修正，不能保留上一套A，只換這套B。這份可另存的A、B與設定稱**adapter（適配器）**；它的意義依賴原基模W，最後使用的是`W+ΔW`。

先用手設數字檢查切換，不訓練文風。第一套B全部0.1；第二套把A設0.2、B設−0.1。兩套rank與alpha相同，下面只檢查矩陣備份和還原。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch
from torch import nn
from tiny_perceptron.alignment import LoRALinear

torch.manual_seed(0)
layer = LoRALinear(nn.Linear(4, 3), rank=2, alpha=2)
with torch.no_grad():
    layer.b.fill_(0.1)
    adapter_a = {"a": layer.a.clone(), "b": layer.b.clone()}
    layer.a.fill_(0.2)
    layer.b.fill_(-0.1)
    adapter_b = {"a": layer.a.clone(), "b": layer.b.clone()}
    layer.a.copy_(adapter_a["a"])
    layer.b.copy_(adapter_a["b"])
print("切回A的B平均", round(layer.b.mean().item(), 2))
print("A兩個矩陣都恢復", torch.equal(layer.a, adapter_a["a"]) and torch.equal(layer.b, adapter_a["b"]))

切回A的B平均 0.1
A兩個矩陣都恢復 True


`no_grad`用於手動設定，`fill_`填入指定值。`clone()`複製數字作備份，`copy_`將備份寫回原位置。最後輸出B平均0.1，以及「兩個矩陣都恢復 True」。只看平均值不足以確認每格與另一個矩陣，才加上兩次`torch.equal`。

實際檔案還須保存基模版本、插入層名、rank、alpha與縮放公式。例如矩陣對了，但alpha從2誤用成4，修正仍會放大兩倍；形狀相同也不代表另一份基模能賦予修正同樣意義。

若部署時不需切換，可以將指定ΔW合入W，保存一份完整模型。合併後不能再套同一adapter，否則重複加入修正。浮點加總順序改變也可能產生很小的數值差，應在同一輸入上核對。

練習刪掉恢復A的那一行。B平均仍0.1，完整還原卻變`False`；這個反例說明為什麼「看見一個數字恢復」與「整套恢復」不同。

<details>
<summary>補充：既有實驗的條件與完整紀錄</summary>

上一節兩套真訓練的adapter也做了「短答→比喻→短答」檢查。在同一個`1+1=?`對話前文上，比較所有位置的候選分數，切回短答前後的最大差為0；短答與比喻版的分數本身則確實不同。這確認了這一次整套切換能恢復，並沒有測到模型對所有新題都有好文風。

若不需要繼續切換，也能把指定修正合進原矩陣，另存一份完整模型：W變成W＋ΔW，推論時就不用再載那套adapter。在同一前文上，比喻adapter與合併模型的候選分數最大差約0.00000668，落在這次設定的0.0001容許值內；浮點計算改了加總順序，通常不保證逐位完全相等。合併檔已含修正，不能再把同一adapter加上一次。實際載入與合併檔的用法見[T.5](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.5)。

</details>
